This is a "stronger" CNN implementation for solving the MNIST multi-class classification task. With this, you should stably reach something **above 99% of accuracy**. More could be done, but we stop here as this is students-readable and educationally useful.

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    Activation,
    MaxPooling2D,
    Dropout,
    Flatten,
    Dense
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [ ]:
X_train = X_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

In [ ]:
y_train_OHE = to_categorical(y_train, num_classes=10)
y_test_OHE = to_categorical(y_test, num_classes=10)

In [ ]:
# data augmentation: apply small random transformations help the model generalize better

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.10,
    width_shift_range=0.10,
    height_shift_range=0.10
)
datagen.fit(X_train)

In [ ]:
model = Sequential()

model.add(Input(shape=(28, 28, 1)))

# First convolutional block
model.add(Conv2D(32, kernel_size=3, padding="same"))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(Conv2D(32, kernel_size=3, padding="same"))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(MaxPooling2D(pool_size=2))
model.add(Dropout(0.25))


# Second convolutional block
model.add(Conv2D(64, kernel_size=3, padding="same"))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(Conv2D(64, kernel_size=3, padding="same"))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(MaxPooling2D(pool_size=2))
model.add(Dropout(0.25))


# Third convolutional block
model.add(Conv2D(128, kernel_size=3, padding="same"))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(Dropout(0.25))


# Dense classifier
model.add(Flatten())

model.add(Dense(256))
model.add(BatchNormalization())
model.add(Activation("relu"))
model.add(Dropout(0.50))

model.add(Dense(10, activation="softmax"))


In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.summary()

In [ ]:
callbacks = [
    ReduceLROnPlateau(
        monitor="val_accuracy",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

In [ ]:
batch_size = 128
epochs = 15

history = model.fit(
    datagen.flow(X_train, y_train_OHE, batch_size=batch_size),
    epochs=epochs,
    validation_data=(X_test, y_test_OHE),
    callbacks=callbacks
)

In [ ]:
print(len(history.history["loss"]))
print(len(history.history["accuracy"]))

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test_OHE, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")